# Earth pipeline: tributary-canyon detection in the Grand Canyon

This notebook is the cleaned, ordered record of the steps that produced the model and the catalogue in this repository
(originally run step by step in Google Colab with a T4 GPU). Cells are kept as they were run; only paths were changed.

* Heavy intermediate files (imagery, DEM, YOLO tiles, training runs) are written under the repository folders
  `data/`, `yolo_dataset/`, `outputs/` and are ignored by git. The small final results are copied into `results/`.
* Step 8 includes manual work in QGIS; the reviewed reference polygons are provided in `annotations/`.
* You do **not** need to run this notebook to use the app. The trained model is in `models/seg_rgb_v2/best.pt`.
* Training needs a GPU (about 10–20 minutes on a Colab T4 per model). Step 18 takes about 2–3 hours on Colab.


In [ ]:
from pathlib import Path
import os

REPO_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TEMP_DIR = Path(os.environ.get('CANYONSCOPE_TEMP', '/tmp/canyonscope'))
TEMP_DIR.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO_DIR, '| temporary files:', TEMP_DIR)


## 1. Project folders

In [ ]:
from pathlib import Path


PROJECT_DIR = REPO_DIR

folders = [
    'data/raw',
    'data/processed',
    'annotations',
    'yolo_dataset',
    'models',
    'outputs',
    'notebooks',
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print('Project folder:', PROJECT_DIR)
print('Folders created:')
for folder in folders:
    print(' -', folder, '| exists:', (PROJECT_DIR / folder).exists())


## 2. Find a clear Sentinel-2 L2A scene (Microsoft Planetary Computer, June–August 2026, < 10% cloud)

In [ ]:
# Packages: pip install -r requirements.txt

from pathlib import Path
import pandas as pd
import planetary_computer
from pystac_client import Client
from shapely.geometry import box, shape

PROJECT_DIR = REPO_DIR
if not PROJECT_DIR.exists():
    raise FileNotFoundError('Project folder not found. Run Step 1 again.')

BBOX = [-112.45, 35.95, -111.85, 36.35]
DATE_RANGE = '2026-06-01/2026-08-31'
MAX_CLOUD = 10

study_area = box(*BBOX)

catalog = Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)

search = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX,
    datetime=DATE_RANGE,
    query={'eo:cloud_cover': {'lt': MAX_CLOUD}},
)
items = list(search.items())

rows = []
for item in items:
    footprint = shape(item.geometry)
    coverage = footprint.intersection(study_area).area / study_area.area * 100
    rows.append({
        'date': item.datetime.date(),
        'scene_id': item.id,
        'tile': item.properties.get('s2:mgrs_tile'),
        'cloud_percent': round(item.properties.get('eo:cloud_cover'), 2),
        'coverage_percent': round(coverage, 1),
    })

print('Scenes found:', len(rows))

if rows:
    results = pd.DataFrame(rows).sort_values(
        ['coverage_percent', 'cloud_percent'],
        ascending=[False, True],
    )
    output_file = PROJECT_DIR / 'data/raw/sentinel2_candidates.csv'
    results.to_csv(output_file, index=False)
    print('Saved to:', output_file)
    print(results[['date', 'tile', 'cloud_percent', 'coverage_percent']].head(15).to_string(index=False))
else:
    print('No scenes found. Send me this result.')

from datetime import date
from shapely.ops import unary_union

if 'items' not in globals():
    raise NameError('Session was reset. Run Step 2 again, then run this cell.')

TARGET_DATE = date(2026, 6, 21)

items_by_date = {}
for item in items:
    scene_date = item.datetime.date()
    items_by_date.setdefault(scene_date, []).append(item)

date_rows = []
for scene_date, day_items in items_by_date.items():
    footprints = [shape(item.geometry) for item in day_items]
    combined_area = unary_union(footprints).intersection(study_area)
    covered_percent = combined_area.area / study_area.area * 100

    tiles = sorted({item.properties.get('s2:mgrs_tile') for item in day_items})
    highest_cloud = max(item.properties.get('eo:cloud_cover') for item in day_items)

    date_rows.append({
        'date': scene_date,
        'scene_count': len(day_items),
        'tiles': ', '.join(tiles),
        'highest_cloud_percent': round(highest_cloud, 2),
        'covered_percent': round(covered_percent, 1),
        'days_from_jun21': abs((scene_date - TARGET_DATE).days),
    })

date_table = pd.DataFrame(date_rows).sort_values(
    ['covered_percent', 'highest_cloud_percent', 'days_from_jun21'],
    ascending=[False, True, True],
)

output_file = PROJECT_DIR / 'data/raw/sentinel2_dates.csv'
date_table.to_csv(output_file, index=False)

print('Dates found:', len(date_table))
print('Saved to:', output_file)
print(date_table.to_string(index=False))


## 3. Download the Sentinel-2 bands for the pilot study area (bbox −112.45, 35.95, −111.85, 36.35)

In [ ]:
# Packages: pip install -r requirements.txt

import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.warp import transform_bounds

SELECTED_DATE = date(2026, 6, 20)
BANDS = ['B02', 'B03', 'B04', 'B08']
BAND_NAMES = ['blue', 'green', 'red', 'nir']

selected_items = [planetary_computer.sign(item) for item in items_by_date[SELECTED_DATE]]

crs_values = set()
for item in selected_items:
    with rasterio.open(item.assets['B04'].href) as src:
        crs_values.add(src.crs.to_string())

print('Scene CRS:', crs_values)
if len(crs_values) != 1:
    raise ValueError('Scenes use different CRS values. Send me this result.')

target_crs = crs_values.pop()
utm_bounds = transform_bounds('EPSG:4326', target_crs, *BBOX, densify_pts=21)

band_arrays = []
output_transform = None

for band in BANDS:
    print('Reading band:', band)
    datasets = [rasterio.open(item.assets[band].href) for item in selected_items]
    mosaic, output_transform = merge(
        datasets,
        bounds=utm_bounds,
        res=10,
        nodata=0,
        method='first',
    )
    for dataset in datasets:
        dataset.close()
    band_arrays.append(mosaic[0])

image = np.stack(band_arrays)

output_file = PROJECT_DIR / 'data/raw/sentinel2_20260620_study_area.tif'
profile = {
    'driver': 'GTiff',
    'height': image.shape[1],
    'width': image.shape[2],
    'count': image.shape[0],
    'dtype': image.dtype,
    'crs': target_crs,
    'transform': output_transform,
    'nodata': 0,
    'compress': 'deflate',
    'tiled': True,
}

with rasterio.open(output_file, 'w', **profile) as dst:
    dst.write(image)
    for index, name in enumerate(BAND_NAMES, start=1):
        dst.set_band_description(index, name)

with rasterio.open(output_file) as src:
    saved = src.read()
    print('Saved to:', output_file)
    print('Bands:', src.count, src.descriptions)
    print('Size (height, width):', src.height, src.width)
    print('CRS:', src.crs)
    print('Pixel size:', src.res)
    print('Empty pixels percent:', round((saved == 0).any(axis=0).mean() * 100, 3))
    for name, band_data in zip(BAND_NAMES, saved):
        valid = band_data[band_data > 0]
        print(name, '| min:', valid.min(), '| max:', valid.max(), '| mean:', round(valid.mean(), 1))

print('File size MB:', round(output_file.stat().st_size / 1024 / 1024, 1))


## 4. Download USGS 3DEP 10 m DEM on the same grid and compute slope, aspect and hillshade

In [ ]:
# Packages: pip install -r requirements.txt

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import planetary_computer
import rasterio
from pystac_client import Client
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT

PROJECT_DIR = REPO_DIR
BBOX = [-112.45, 35.95, -111.85, 36.35]
image_file = PROJECT_DIR / 'data/raw/sentinel2_20260620_study_area.tif'
dem_file = PROJECT_DIR / 'data/raw/dem_10m_study_area.tif'
dem_preview_file = PROJECT_DIR / 'outputs/dem_preview.png'

catalog = Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)

search = catalog.search(collections=['3dep-seamless'], bbox=BBOX)
all_dem_items = list(search.items())
dem_items = [item for item in all_dem_items if item.properties.get('gsd') == 10]

print('DEM items found:', len(all_dem_items))
print('10 m items:', [item.id for item in dem_items])

if not dem_items:
    raise ValueError('No 10 m DEM items found. Send me this result.')

with rasterio.open(image_file) as reference:
    ref_profile = reference.profile
    ref_crs = reference.crs
    ref_transform = reference.transform
    ref_width = reference.width
    ref_height = reference.height
    ref_bounds = reference.bounds

dem = np.full((ref_height, ref_width), np.nan, dtype='float32')

for item in dem_items:
    print('Reading:', item.id)
    with rasterio.open(item.assets['data'].href) as src:
        with WarpedVRT(
            src,
            crs=ref_crs,
            transform=ref_transform,
            width=ref_width,
            height=ref_height,
            resampling=Resampling.bilinear,
        ) as vrt:
            tile = vrt.read(1).astype('float32')
    tile[tile <= 0] = np.nan
    dem = np.where(np.isnan(dem), tile, dem)

profile = ref_profile.copy()
profile.update(count=1, dtype='float32', nodata=np.nan, compress='deflate', predictor=3, tiled=True)

with rasterio.open(dem_file, 'w', **profile) as dst:
    dst.write(dem, 1)
    dst.set_band_description(1, 'elevation_m')

with rasterio.open(dem_file) as src:
    saved_dem = src.read(1)
    print('Saved to:', dem_file)
    print('Size (height, width):', src.height, src.width)
    print('CRS:', src.crs)
    print('Pixel size:', src.res)
    print('Same grid as Sentinel-2:', src.transform == ref_transform and src.width == ref_width and src.height == ref_height)

print('Empty pixels percent:', round(np.isnan(saved_dem).mean() * 100, 3))
print('Elevation min (m):', round(float(np.nanmin(saved_dem)), 1))
print('Elevation max (m):', round(float(np.nanmax(saved_dem)), 1))
print('Elevation mean (m):', round(float(np.nanmean(saved_dem)), 1))
print('File size MB:', round(dem_file.stat().st_size / 1024 / 1024, 1))

extent = [ref_bounds.left / 1000, ref_bounds.right / 1000, ref_bounds.bottom / 1000, ref_bounds.top / 1000]
plt.figure(figsize=(12, 10))
plt.imshow(saved_dem[::5, ::5], cmap='terrain', extent=extent)
plt.colorbar(label='Elevation (m)')
plt.title('USGS 3DEP DEM - 10 m')
plt.xlabel('UTM Easting (km)')
plt.ylabel('UTM Northing (km)')
plt.savefig(dem_preview_file, dpi=150, bbox_inches='tight')
plt.show()

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window

PROJECT_DIR = REPO_DIR
image_file = PROJECT_DIR / 'data/raw/sentinel2_20260620_study_area.tif'
dem_file = PROJECT_DIR / 'data/raw/dem_10m_study_area.tif'
output_file = PROJECT_DIR / 'data/processed/dem_derivatives.tif'
check_file = PROJECT_DIR / 'outputs/alignment_check.png'

PIXEL_SIZE = 10
SUN_AZIMUTH = 315
SUN_ALTITUDE = 45

with rasterio.open(dem_file) as src:
    dem = src.read(1)
    profile = src.profile
    transform = src.transform

row_min, col_min = np.unravel_index(np.nanargmin(dem), dem.shape)
x_min, y_min = rasterio.transform.xy(transform, row_min, col_min)
print('Lowest point (km):', round(x_min / 1000, 1), round(y_min / 1000, 1))
print('Lowest elevation (m):', round(float(dem[row_min, col_min]), 1))

dz_row, dz_col = np.gradient(dem, PIXEL_SIZE)
dz_east = dz_col
dz_north = -dz_row

slope = np.degrees(np.arctan(np.sqrt(dz_east**2 + dz_north**2)))
aspect = np.degrees(np.arctan2(-dz_east, -dz_north)) % 360

azimuth = np.radians(SUN_AZIMUTH)
altitude = np.radians(SUN_ALTITUDE)
sun_east = np.sin(azimuth) * np.cos(altitude)
sun_north = np.cos(azimuth) * np.cos(altitude)
sun_up = np.sin(altitude)

normal_length = np.sqrt(dz_east**2 + dz_north**2 + 1)
hillshade = (-dz_east * sun_east - dz_north * sun_north + sun_up) / normal_length
hillshade = np.clip(hillshade, 0, 1)

layers = np.stack([slope, aspect, hillshade]).astype('float32')
profile.update(count=3, dtype='float32', nodata=np.nan, compress='deflate', predictor=3, tiled=True)

with rasterio.open(output_file, 'w', **profile) as dst:
    dst.write(layers)
    dst.set_band_description(1, 'slope_degrees')
    dst.set_band_description(2, 'aspect_degrees')
    dst.set_band_description(3, 'hillshade_0_1')

print('Saved to:', output_file)
print('Slope min/max:', round(float(np.nanmin(slope)), 1), round(float(np.nanmax(slope)), 1))
print('Slope mean:', round(float(np.nanmean(slope)), 1))
print('Hillshade min/max:', round(float(np.nanmin(hillshade)), 2), round(float(np.nanmax(hillshade)), 2))

window = Window(2000, 1500, 1000, 1000)
with rasterio.open(image_file) as src:
    rgb_raw = src.read([3, 2, 1], window=window).astype('float32')

reflectance = (rgb_raw - 1000) / 10000
rgb = np.zeros((1000, 1000, 3), dtype='float32')
for i in range(3):
    low, high = np.percentile(reflectance[i], [2, 98])
    rgb[:, :, i] = np.clip((reflectance[i] - low) / (high - low), 0, 1)

row_start, col_start = 1500, 2000
hillshade_crop = hillshade[row_start:row_start + 1000, col_start:col_start + 1000]
slope_crop = slope[row_start:row_start + 1000, col_start:col_start + 1000]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(rgb)
axes[0].set_title('Sentinel-2')
axes[1].imshow(hillshade_crop, cmap='gray')
axes[1].set_title('Hillshade')
slope_plot = axes[2].imshow(slope_crop, cmap='magma', vmin=0, vmax=60)
axes[2].set_title('Slope (degrees)')
fig.colorbar(slope_plot, ax=axes[2], fraction=0.046)
plt.savefig(check_file, dpi=150, bbox_inches='tight')
plt.show()

print('Saved to:', check_file)


## 5. 8-bit RGB and terrain images + stretch settings (used again by the live detector)

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling

PROJECT_DIR = REPO_DIR
image_file = PROJECT_DIR / 'data/raw/sentinel2_20260620_study_area.tif'
dem_file = PROJECT_DIR / 'data/raw/dem_10m_study_area.tif'
derivatives_file = PROJECT_DIR / 'data/processed/dem_derivatives.tif'

rgb_output = PROJECT_DIR / 'data/processed/rgb_8bit.tif'
terrain_output = PROJECT_DIR / 'data/processed/terrain_8bit.tif'
settings_output = PROJECT_DIR / 'data/processed/stretch_settings.json'
preview_output = PROJECT_DIR / 'outputs/model_inputs_preview.png'

OFFSET = 1000
SLOPE_MAX = 60
ELEVATION_MIN = 300
ELEVATION_MAX = 3000


def to_8bit(values, low, high, valid_mask):
    scaled = np.clip((values - low) / (high - low), 0, 1)
    result = np.rint(scaled * 254 + 1).astype('uint8')
    result[~valid_mask] = 0
    return result


def save_8bit(path, array, band_names, base_profile):
    profile = base_profile.copy()
    profile.update(
        count=3,
        dtype='uint8',
        nodata=0,
        compress='deflate',
        predictor=2,
        tiled=True,
    )
    with rasterio.open(path, 'w', **profile) as dst:
        dst.write(array)
        for index, name in enumerate(band_names, start=1):
            dst.set_band_description(index, name)
    with rasterio.open(path, 'r+') as dst:
        dst.build_overviews([2, 4, 8, 16], Resampling.average)


with rasterio.open(image_file) as src:
    sentinel = src.read([3, 2, 1]).astype('float32')
    base_profile = src.profile

with rasterio.open(dem_file) as src:
    elevation = src.read(1)

with rasterio.open(derivatives_file) as src:
    slope = src.read(1)
    hillshade = src.read(3)

rgb_valid = (sentinel > 0).all(axis=0)
reflectance = (sentinel - OFFSET) / 10000

rgb_limits = {}
rgb_bands = []
for name, band in zip(['red', 'green', 'blue'], reflectance):
    low, high = np.percentile(band[rgb_valid][::5], [2, 98])
    rgb_limits[name] = {'low': float(low), 'high': float(high)}
    rgb_bands.append(to_8bit(band, low, high, rgb_valid))

rgb_8bit = np.stack(rgb_bands)

terrain_valid = np.isfinite(elevation) & np.isfinite(slope) & np.isfinite(hillshade)
terrain_8bit = np.stack([
    to_8bit(hillshade, 0, 1, terrain_valid),
    to_8bit(slope, 0, SLOPE_MAX, terrain_valid),
    to_8bit(elevation, ELEVATION_MIN, ELEVATION_MAX, terrain_valid),
])

save_8bit(rgb_output, rgb_8bit, ['red', 'green', 'blue'], base_profile)
save_8bit(terrain_output, terrain_8bit, ['hillshade', 'slope', 'elevation'], base_profile)

settings = {
    'offset': OFFSET,
    'rgb_reflectance_limits': rgb_limits,
    'hillshade_limits': [0, 1],
    'slope_limits_degrees': [0, SLOPE_MAX],
    'elevation_limits_m': [ELEVATION_MIN, ELEVATION_MAX],
}
settings_output.write_text(json.dumps(settings, indent=2))

for path in [rgb_output, terrain_output]:
    with rasterio.open(path) as src:
        data = src.read()
        print(path.name)
        print('  Size:', src.height, src.width, '| Bands:', src.count, src.descriptions)
        print('  CRS:', src.crs, '| Pixel size:', src.res, '| Type:', src.dtypes[0])
        print('  Empty pixels percent:', round((data == 0).any(axis=0).mean() * 100, 3))
        print('  Overviews:', src.overviews(1))
        print('  File size MB:', round(path.stat().st_size / 1024 / 1024, 1))

print('Same grid:', rgb_8bit.shape == terrain_8bit.shape)
print('Settings saved to:', settings_output)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(np.moveaxis(rgb_8bit[:, ::5, ::5], 0, -1))
axes[0].set_title('RGB input')
axes[1].imshow(np.moveaxis(terrain_8bit[:, ::5, ::5], 0, -1))
axes[1].set_title('Terrain input: hillshade, slope, elevation')
for axis in axes:
    axis.set_axis_off()
plt.savefig(preview_output, dpi=150, bbox_inches='tight')
plt.show()

print('Preview saved to:', preview_output)


## 6. Spatial annotation zones (train / validation / test strips)

In [ ]:
# Packages: pip install -r requirements.txt

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from shapely.geometry import box

PROJECT_DIR = REPO_DIR
image_file = PROJECT_DIR / 'data/processed/rgb_8bit.tif'
annotation_file = PROJECT_DIR / 'annotations/canyons_ground_truth.gpkg'
zones_file = PROJECT_DIR / 'annotations/annotation_zones.gpkg'
preview_file = PROJECT_DIR / 'outputs/annotation_zones_preview.png'

with rasterio.open(image_file) as src:
    bounds = src.bounds
    crs = src.crs
    rgb = src.read(out_shape=(3, src.height // 5, src.width // 5))

zone_ranges = [
    ('zone_1', 'test', bounds.left, 382000),
    ('zone_2', 'train', 383500, 396000),
    ('zone_3', 'train', 397500, 410500),
    ('zone_4', 'validation', 412000, bounds.right),
]

zones = gpd.GeoDataFrame(
    {
        'zone_name': [zone[0] for zone in zone_ranges],
        'split': [zone[1] for zone in zone_ranges],
    },
    geometry=[box(zone[2], bounds.bottom, zone[3], bounds.top) for zone in zone_ranges],
    crs=crs,
)

zones.to_file(zones_file, driver='GPKG', layer='annotation_zones')
zones['width_km'] = (zones.geometry.bounds['maxx'] - zones.geometry.bounds['minx']) / 1000
zones['area_km2'] = zones.geometry.area / 1_000_000

print('Saved to:', zones_file)
print(zones[['zone_name', 'split', 'width_km', 'area_km2']].round(1).to_string(index=False))

canyons = gpd.read_file(annotation_file)
matches = gpd.sjoin(
    canyons[['canyon_name', 'geometry']],
    zones[['zone_name', 'split', 'geometry']],
    how='left',
    predicate='intersects',
)
print('Canyon zones:')
print(matches[['canyon_name', 'zone_name', 'split']].to_string(index=False))

extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
fig, axis = plt.subplots(figsize=(13, 10))
axis.imshow(np.moveaxis(rgb, 0, -1), extent=extent)
zones.boundary.plot(ax=axis, color='yellow', linewidth=2)
canyons.boundary.plot(ax=axis, color='cyan', linewidth=2)

for _, zone in zones.iterrows():
    center = zone.geometry.centroid
    axis.text(
        center.x,
        bounds.top - 2500,
        f'{zone.zone_name}\n{zone.split}',
        color='white',
        ha='center',
        fontsize=12,
        bbox={'facecolor': 'black', 'alpha': 0.6},
    )

axis.set_title('Annotation zones')
plt.savefig(preview_file, dpi=150, bbox_inches='tight')
plt.show()

print('Preview saved to:', preview_file)

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds

PROJECT_DIR = REPO_DIR
dem_file = PROJECT_DIR / 'data/raw/dem_10m_study_area.tif'
output_file = PROJECT_DIR / 'outputs/C002_profile.png'

LEFT, BOTTOM, RIGHT, TOP = 402000, 3988000, 409000, 3996000
PROFILE_Y = 3992000
CENTER_X = 405250

with rasterio.open(dem_file) as src:
    window = from_bounds(LEFT, BOTTOM, RIGHT, TOP, transform=src.transform)
    dem = src.read(1, window=window)
    transform = src.window_transform(window)

rows, cols = dem.shape
x_values = transform.c + (np.arange(cols) + 0.5) * transform.a
y_values = transform.f + (np.arange(rows) + 0.5) * transform.e

profile_row = int(np.argmin(np.abs(y_values - PROFILE_Y)))
profile = dem[profile_row]

west = x_values < CENTER_X
east = x_values > CENTER_X
center_index = int(np.argmin(np.abs(x_values - CENTER_X)))

print('Center elevation (m):', round(float(profile[center_index]), 1))
print('Lowest west of center (m):', round(float(np.nanmin(profile[west])), 1))
print('Lowest east of center (m):', round(float(np.nanmin(profile[east])), 1))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

levels = np.arange(600, 2900, 50)
contours = axes[0].contour(x_values, y_values, dem, levels=levels, colors='black', linewidths=0.5)
axes[0].clabel(contours, levels[::4], fmt='%d', fontsize=8)
axes[0].axhline(PROFILE_Y, color='red', linewidth=2)
axes[0].scatter([CENTER_X], [PROFILE_Y], color='red', s=50)
axes[0].set_aspect('equal')
axes[0].set_title('Contours every 50 m (red line = profile)')

axes[1].plot(x_values, profile, color='brown', linewidth=2)
axes[1].axvline(CENTER_X, color='red', linestyle='--')
axes[1].set_title('Elevation profile through C002 (west to east)')
axes[1].set_xlabel('UTM Easting (m)')
axes[1].set_ylabel('Elevation (m)')
axes[1].grid(True)

plt.savefig(output_file, dpi=150, bbox_inches='tight')
plt.show()

print('Saved to:', output_file)


## 7. Valley–ridge layer (DEM minus its 1 km mean) used to guide annotation

In [ ]:
# Packages: pip install -r requirements.txt

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import from_bounds
from scipy.ndimage import uniform_filter

PROJECT_DIR = REPO_DIR
dem_file = PROJECT_DIR / 'data/raw/dem_10m_study_area.tif'
annotation_file = PROJECT_DIR / 'annotations/canyons_ground_truth.gpkg'
output_file = PROJECT_DIR / 'data/processed/valley_ridge_8bit.tif'
preview_file = PROJECT_DIR / 'outputs/valley_ridge_preview.png'

WINDOW_PIXELS = 101
LIMIT_METERS = 200

with rasterio.open(dem_file) as src:
    dem = src.read(1)
    profile = src.profile

local_mean = uniform_filter(dem, size=WINDOW_PIXELS, mode='nearest')
difference = dem - local_mean
scaled = np.clip(difference / LIMIT_METERS, -1, 1)

red = 255 * (1 + np.minimum(scaled, 0))
green = 255 * (1 - np.abs(scaled))
blue = 255 * (1 - np.maximum(scaled, 0))
image = np.stack([red, green, blue]).round().astype('uint8')

profile.update(count=3, dtype='uint8', nodata=None, compress='deflate', predictor=2, tiled=True)

with rasterio.open(output_file, 'w', **profile) as dst:
    dst.write(image)

with rasterio.open(output_file, 'r+') as dst:
    dst.build_overviews([2, 4, 8, 16], Resampling.average)

print('Saved to:', output_file)
print('File size MB:', round(output_file.stat().st_size / 1024 / 1024, 1))

LEFT, BOTTOM, RIGHT, TOP = 402000, 3988000, 409000, 3996000
with rasterio.open(output_file) as src:
    window = from_bounds(LEFT, BOTTOM, RIGHT, TOP, transform=src.transform)
    crop = src.read(window=window)

canyons = gpd.read_file(annotation_file)

fig, axis = plt.subplots(figsize=(11, 11))
axis.imshow(np.moveaxis(crop, 0, -1), extent=[LEFT, RIGHT, BOTTOM, TOP])
canyons.boundary.plot(ax=axis, color='black', linewidth=2)
axis.set_xlim(LEFT, RIGHT)
axis.set_ylim(BOTTOM, TOP)
axis.set_title('Blue = valley, red = ridge, black = C001 and C002 area')
plt.savefig(preview_file, dpi=150, bbox_inches='tight')
plt.show()

print('Preview saved to:', preview_file)


## 8. Candidate canyon polygons from the DEM (watershed), reviewed and corrected by hand in QGIS

The candidates were reviewed and edited manually in QGIS (CRS EPSG:32612) on top of the RGB, valley–ridge and contour layers. The reviewed reference set is `annotations/canyons_ground_truth_v8.gpkg` (71 polygons, 457 km²).

In [ ]:
# Packages: pip install -r requirements.txt

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio.features import shapes
from scipy.ndimage import gaussian_filter, maximum_filter, minimum_filter, uniform_filter, binary_fill_holes
from shapely.geometry import shape
from shapely.ops import unary_union
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects
from skimage.segmentation import watershed

PROJECT_DIR = REPO_DIR
dem_file = PROJECT_DIR / 'data/raw/dem_10m_study_area.tif'
image_file = PROJECT_DIR / 'data/processed/rgb_8bit.tif'
annotation_file = PROJECT_DIR / 'annotations/canyons_ground_truth.gpkg'
output_file = PROJECT_DIR / 'annotations/candidates_v1.gpkg'
preview_file = PROJECT_DIR / 'outputs/candidates_v1.png'

RELIEF_WINDOW = 101
MIN_RELIEF_M = 250
STOP_ELEVATION_M = 1000
VALLEY_THRESHOLD_M = -50
MIN_VALLEY_PIXELS = 15000
MIN_AREA_KM2 = 2
MAX_AREA_KM2 = 25
SIMPLIFY_M = 30

with rasterio.open(dem_file) as src:
    dem = src.read(1)
    transform = src.transform
    crs = src.crs
    bounds = src.bounds

relief = maximum_filter(dem, size=RELIEF_WINDOW) - minimum_filter(dem, size=RELIEF_WINDOW)
canyon_mask = binary_fill_holes(relief >= MIN_RELIEF_M) & (dem >= STOP_ELEVATION_M)

smooth_dem = gaussian_filter(dem, sigma=8)
position = smooth_dem - uniform_filter(smooth_dem, size=RELIEF_WINDOW, mode='nearest')

valleys = (position <= VALLEY_THRESHOLD_M) & canyon_mask
valleys = remove_small_objects(valleys, min_size=MIN_VALLEY_PIXELS)
markers = label(valleys, connectivity=2)
print('Valley groups:', int(markers.max()))

regions = watershed(smooth_dem, markers=markers, mask=canyon_mask)

records = []
for geometry, value in shapes(regions.astype('int32'), mask=regions > 0, transform=transform):
    polygon = shape(geometry).buffer(0)
    if polygon.geom_type == 'MultiPolygon':
        polygon = max(polygon.geoms, key=lambda part: part.area)
    polygon = polygon.simplify(SIMPLIFY_M).buffer(0)
    area_km2 = polygon.area / 1_000_000
    if MIN_AREA_KM2 <= area_km2 <= MAX_AREA_KM2:
        records.append({'area_km2': round(area_km2, 2), 'geometry': polygon})

candidates = gpd.GeoDataFrame(records, crs=crs)

canyons = gpd.read_file(annotation_file)
reviewed_area = unary_union(canyons.geometry)
overlap_share = candidates.geometry.intersection(reviewed_area).area / candidates.geometry.area
candidates = candidates[overlap_share < 0.3].reset_index(drop=True)

candidates['cand_id'] = [f'K{i + 1:03d}' for i in range(len(candidates))]
candidates['canyon_type'] = 'primary'
candidates['certainty'] = 'unchecked'
candidates['notes'] = None
candidates = candidates[['cand_id', 'canyon_type', 'certainty', 'notes', 'area_km2', 'geometry']]

if output_file.exists():
    output_file.unlink()
candidates.to_file(output_file, driver='GPKG', layer='candidates_v1')

print('Candidates:', len(candidates))
print('Area min/mean/max km2:', candidates['area_km2'].min(), round(candidates['area_km2'].mean(), 2), candidates['area_km2'].max())

with rasterio.open(image_file) as src:
    preview = src.read(out_shape=(3, src.height // 5, src.width // 5))

fig, axis = plt.subplots(figsize=(15, 12))
axis.imshow(np.moveaxis(preview, 0, -1), extent=[bounds.left, bounds.right, bounds.bottom, bounds.top])
candidates.boundary.plot(ax=axis, color='cyan', linewidth=1.2)
canyons.boundary.plot(ax=axis, color='yellow', linewidth=2)
axis.set_title('Cyan = automatic candidates, yellow = reviewed canyons')
plt.savefig(preview_file, dpi=150, bbox_inches='tight')
plt.show()

print('Saved to:', output_file)

from rasterio.features import rasterize

required = ['dem', 'smooth_dem', 'position', 'canyon_mask', 'canyons', 'transform', 'crs', 'bounds', 'preview']
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f'Session was reset. Run Step 28 again first. Missing: {missing}')

THRESHOLDS = [-50, -90, -130, -170]
MIN_VALLEY_PIXELS = 5000
MIN_AREA_KM2 = 2
MAX_AREA_KM2 = 20
SIMPLIFY_M = 30
output_file = PROJECT_DIR / 'annotations/candidates_v2.gpkg'
preview_file = PROJECT_DIR / 'outputs/candidates_v2.png'

reviewed_mask = rasterize(
    [(geometry, 1) for geometry in canyons.geometry],
    out_shape=dem.shape,
    transform=transform,
    fill=0,
    dtype='uint8',
).astype(bool)

remaining = canyon_mask & ~reviewed_mask
records = []

for threshold in THRESHOLDS:
    valleys = (position <= threshold) & remaining
    valleys = remove_small_objects(valleys, min_size=MIN_VALLEY_PIXELS)
    markers = label(valleys, connectivity=2)

    if markers.max() == 0:
        print('Threshold', threshold, '| no valley groups')
        continue

    regions = watershed(smooth_dem, markers=markers, mask=remaining)
    area_km2 = np.bincount(regions.ravel()) * 0.0001
    good = (area_km2 >= MIN_AREA_KM2) & (area_km2 <= MAX_AREA_KM2)
    good[0] = False
    accepted = good[regions]

    accepted_count = 0
    for geometry, value in shapes(regions.astype('int32'), mask=accepted, transform=transform):
        polygon = shape(geometry).buffer(0)
        if polygon.geom_type == 'MultiPolygon':
            polygon = max(polygon.geoms, key=lambda part: part.area)
        polygon = polygon.simplify(SIMPLIFY_M).buffer(0)
        polygon_area = polygon.area / 1_000_000
        if MIN_AREA_KM2 <= polygon_area <= MAX_AREA_KM2:
            records.append({'area_km2': round(polygon_area, 2), 'source_threshold': threshold, 'geometry': polygon})
            accepted_count += 1

    remaining = remaining & ~accepted
    print('Threshold', threshold, '| groups:', int(markers.max()), '| accepted:', accepted_count)

candidates = gpd.GeoDataFrame(records, crs=crs)
candidates['cand_id'] = [f'K{i + 1:03d}' for i in range(len(candidates))]
candidates['canyon_type'] = 'primary'
candidates['certainty'] = 'unchecked'
candidates['notes'] = None
candidates = candidates[['cand_id', 'canyon_type', 'certainty', 'notes', 'area_km2', 'source_threshold', 'geometry']]

if output_file.exists():
    output_file.unlink()
candidates.to_file(output_file, driver='GPKG', layer='candidates_v2')

print('Candidates:', len(candidates))
print('Area min/mean/max km2:', candidates['area_km2'].min(), round(candidates['area_km2'].mean(), 2), candidates['area_km2'].max())

fig, axis = plt.subplots(figsize=(15, 12))
axis.imshow(np.moveaxis(preview, 0, -1), extent=[bounds.left, bounds.right, bounds.bottom, bounds.top])
candidates.boundary.plot(ax=axis, color='cyan', linewidth=1.2)
canyons.boundary.plot(ax=axis, color='yellow', linewidth=2)
axis.set_title('Cyan = automatic candidates v2, yellow = reviewed canyons')
plt.savefig(preview_file, dpi=150, bbox_inches='tight')
plt.show()

print('Saved to:', output_file)

MIN_DEPTH_FROM_TOP_M = 60

depth_from_top = maximum_filter(dem, size=RELIEF_WINDOW) - dem
canyon_mask = canyon_mask & (depth_from_top >= MIN_DEPTH_FROM_TOP_M)

print('Canyon mask share percent:', round(float(canyon_mask.mean()) * 100, 1))


## 9. Build the YOLO segmentation dataset (640 px tiles, spatially separated strips)

In [ ]:
# Packages: pip install -r requirements.txt

from pathlib import Path
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from PIL import Image
from rasterio.features import rasterize
from rasterio.windows import Window, bounds as window_bounds
from scipy.ndimage import binary_fill_holes, maximum_filter, minimum_filter
from shapely.geometry import box


PROJECT_DIR = REPO_DIR
dem_file = PROJECT_DIR / 'data/raw/dem_10m_study_area.tif'
truth_file = PROJECT_DIR / 'annotations/canyons_ground_truth_v8.gpkg'
preview_file = PROJECT_DIR / 'outputs/dataset_tile_map.png'

inputs = {
    'full_rgb': PROJECT_DIR / 'data/processed/rgb_8bit.tif',
    'full_terrain': PROJECT_DIR / 'data/processed/terrain_8bit.tif',
}

SPLIT_STRIPS = {
    'train': [(397500, 410500), (383500, 396000)],
    'val': [(412000, float('inf'))],
    'test': [(float('-inf'), 382000)],
}
STEPS = {'train': 128, 'val': 160, 'test': 160}
TILE_SIZE = 640
MIN_LABEL_AREA_M2 = 100000
MAX_UNLABELED_SHARE = 0.25
EMPTY_TILE_SHARE = 0.2
random.seed(42)

truth = gpd.read_file(truth_file)
if len(truth) != 71:
    raise ValueError(f'Version 8 has {len(truth)} canyons, expected 71. Send me this result.')

with rasterio.open(dem_file) as src:
    dem = src.read(1)
    transform = src.transform
    height, width = src.height, src.width
    pixel_size = src.res[0]

relief = maximum_filter(dem, size=101) - minimum_filter(dem, size=101)
canyon_like = binary_fill_holes(relief >= 250) & (dem >= 1000)
canyon_like &= (maximum_filter(dem, size=101) - dem) >= 60

labeled = rasterize(
    [(geometry, 1) for geometry in truth.geometry],
    out_shape=dem.shape,
    transform=transform,
    fill=0,
    dtype='uint8',
).astype(bool)
unlabeled = canyon_like & ~labeled

tile_width = TILE_SIZE * pixel_size


def polygon_parts(geometry):
    if geometry.is_empty:
        return []
    if geometry.geom_type == 'Polygon':
        return [geometry]
    if hasattr(geometry, 'geoms'):
        return [part for part in geometry.geoms if part.geom_type == 'Polygon']
    return []


def label_lines(tile_shape, left, top):
    lines = []
    for geometry in truth.geometry:
        if not geometry.intersects(tile_shape):
            continue
        for part in polygon_parts(geometry.intersection(tile_shape)):
            if part.area < MIN_LABEL_AREA_M2:
                continue
            values = ['0']
            for x, y in list(part.exterior.coords)[:-1]:
                values.append(f'{np.clip((x - left) / tile_width, 0, 1):.6f}')
                values.append(f'{np.clip((top - y) / tile_width, 0, 1):.6f}')
            lines.append(' '.join(values))
    return lines


tiles = []
skipped = 0

for split, strips in SPLIT_STRIPS.items():
    step = STEPS[split]
    split_tiles = []
    for row in range(0, height - TILE_SIZE + 1, step):
        for col in range(0, width - TILE_SIZE + 1, step):
            window = Window(col, row, TILE_SIZE, TILE_SIZE)
            left, bottom, right, top = window_bounds(window, transform)
            if not any(left >= min_x and right <= max_x for min_x, max_x in strips):
                continue
            if unlabeled[row:row + TILE_SIZE, col:col + TILE_SIZE].mean() > MAX_UNLABELED_SHARE:
                skipped += 1
                continue
            lines = label_lines(box(left, bottom, right, top), left, top)
            split_tiles.append({'row': row, 'col': col, 'split': split, 'left': left, 'bottom': bottom, 'lines': lines})

    with_labels = [tile for tile in split_tiles if tile['lines']]
    empty = [tile for tile in split_tiles if not tile['lines']]
    keep_empty = min(len(empty), int(len(with_labels) * EMPTY_TILE_SHARE))
    tiles.extend(with_labels + random.sample(empty, keep_empty))

print('Tiles skipped for unlabeled land:', skipped)

for dataset_name, raster_file in inputs.items():
    dataset_dir = PROJECT_DIR / 'yolo_dataset' / dataset_name
    if dataset_dir.exists():
        shutil.rmtree(dataset_dir)

    for split in SPLIT_STRIPS:
        (dataset_dir / 'images' / split).mkdir(parents=True)
        (dataset_dir / 'labels' / split).mkdir(parents=True)

    with rasterio.open(raster_file) as src:
        for tile in tiles:
            window = Window(tile['col'], tile['row'], TILE_SIZE, TILE_SIZE)
            image = np.moveaxis(src.read(window=window), 0, -1)
            name = f"tile_r{tile['row']:05d}_c{tile['col']:05d}"
            Image.fromarray(image).save(dataset_dir / 'images' / tile['split'] / f'{name}.png')
            text = '\n'.join(tile['lines'])
            (dataset_dir / 'labels' / tile['split'] / f'{name}.txt').write_text(text + ('\n' if text else ''))

    (dataset_dir / 'data.yaml').write_text(
        f'path: {dataset_dir}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: canyon\n'
    )
    print('Dataset created:', dataset_name)

for split in SPLIT_STRIPS:
    split_tiles = [tile for tile in tiles if tile['split'] == split]
    with_labels = sum(1 for tile in split_tiles if tile['lines'])
    shapes_count = sum(len(tile['lines']) for tile in split_tiles)
    print(split, '| tiles:', len(split_tiles), '| with labels:', with_labels, '| empty:', len(split_tiles) - with_labels, '| label shapes:', shapes_count)

colors = {'train': 'lime', 'val': 'gold', 'test': 'red'}
fig, axis = plt.subplots(figsize=(12, 10))
truth.plot(ax=axis, color='lightblue', edgecolor='blue', linewidth=0.5)
for tile in tiles:
    axis.add_patch(plt.Rectangle((tile['left'], tile['bottom']), tile_width, tile_width,
                                 fill=False, edgecolor=colors[tile['split']], linewidth=0.5))
axis.autoscale_view()
axis.set_title('Dataset tiles: green train, yellow val, red test')
axis.set_aspect('equal')
plt.savefig(preview_file, dpi=130, bbox_inches='tight')
plt.show()
print('Saved to:', preview_file)


## 10. Train model v1 (yolo26s-seg)

In [ ]:
# Packages: pip install -r requirements.txt

import csv
import shutil
from pathlib import Path

import torch
from ultralytics import YOLO


if not torch.cuda.is_available():
    raise RuntimeError('GPU is not active. Select T4 GPU, then run again.')

print('GPU:', torch.cuda.get_device_name(0))

PROJECT_DIR = REPO_DIR
LOCAL_DIR = (TEMP_DIR / 'datasets')
RUNS = {
    'seg_rgb_v1': 'full_rgb',
    'seg_terrain_v1': 'full_terrain',
}

results = []

for run_name, dataset_name in RUNS.items():
    print('Starting:', run_name)
    local_dataset = LOCAL_DIR / dataset_name
    if local_dataset.exists():
        shutil.rmtree(local_dataset)

    shutil.copytree(PROJECT_DIR / 'yolo_dataset' / dataset_name, local_dataset)
    data_file = local_dataset / 'data.yaml'
    data_file.write_text(
        f'path: {local_dataset}\n'
        'train: images/train\n'
        'val: images/val\n'
        'test: images/test\n'
        'names:\n'
        '  0: canyon\n'
    )

    model = YOLO('yolo26s-seg.pt')
    model.train(
        data=str(data_file),
        epochs=100,
        patience=30,
        imgsz=640,
        batch=8,
        seed=42,
        workers=2,
        project=str(PROJECT_DIR / 'models'),
        name=run_name,
        exist_ok=True,
        verbose=False,
    )

    best_file = Path(model.trainer.best)
    metrics = YOLO(best_file).val(data=str(data_file), split='val', imgsz=640, verbose=False)

    precision = float(metrics.seg.mp)
    recall = float(metrics.seg.mr)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

    results.append({
        'run': run_name,
        'mask_precision': round(precision, 3),
        'mask_recall': round(recall, 3),
        'mask_f1': round(f1, 3),
        'mask_map50': round(float(metrics.seg.map50), 3),
        'mask_map50_95': round(float(metrics.seg.map), 3),
        'best_file': str(best_file),
    })

output_file = PROJECT_DIR / 'outputs/validation_comparison.csv'
with output_file.open('w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print('Training finished')
for result in results:
    print(result)
print('Saved to:', output_file)


## 11. Confidence-threshold comparison

In [ ]:
# Packages: pip install -r requirements.txt

import csv
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw
from ultralytics import YOLO


PROJECT_DIR = REPO_DIR
MODELS = {
    'rgb': (PROJECT_DIR / 'models/seg_rgb_v1/weights/best.pt', PROJECT_DIR / 'yolo_dataset/full_rgb'),
    'terrain': (PROJECT_DIR / 'models/seg_terrain_v1_final/best.pt', PROJECT_DIR / 'yolo_dataset/full_terrain'),
}
THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.35, 0.50]
IMAGE_SIZE = 640
OUTPUT_FILE = PROJECT_DIR / 'outputs/threshold_comparison.csv'


def polygon_mask(points):
    mask = Image.new('L', (IMAGE_SIZE, IMAGE_SIZE), 0)
    if len(points) >= 3:
        ImageDraw.Draw(mask).polygon([tuple(point) for point in points], fill=1)
    return np.array(mask, dtype=bool)


def truth_masks(label_file):
    masks = []
    for line in label_file.read_text().splitlines():
        points = np.array(line.split()[1:], dtype='float32').reshape(-1, 2) * IMAGE_SIZE
        masks.append(polygon_mask(points))
    return masks


rows = []

for model_name, (model_file, dataset_dir) in MODELS.items():
    if not model_file.exists():
        raise FileNotFoundError(f'Missing model: {model_file}')

    model = YOLO(model_file)
    stored = []

    for image_file in sorted((dataset_dir / 'images/val').glob('*.png')):
        masks = truth_masks(dataset_dir / 'labels/val' / f'{image_file.stem}.txt')
        result = model.predict(str(image_file), conf=min(THRESHOLDS), imgsz=IMAGE_SIZE, verbose=False)[0]
        predictions = []
        if result.masks is not None:
            for polygon, score in zip(result.masks.xy, result.boxes.conf.cpu().numpy()):
                mask = polygon_mask(polygon)
                if mask.any():
                    predictions.append((float(score), mask))
        stored.append((masks, predictions))

    for threshold in THRESHOLDS:
        true_positive = false_positive = false_negative = 0
        truth_count = found_truth = predicted_count = correct_predicted = 0

        for masks, predictions in stored:
            selected = [mask for score, mask in predictions if score >= threshold]
            empty = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)
            truth_union = np.any(masks, axis=0) if masks else empty
            predicted_union = np.any(selected, axis=0) if selected else empty

            true_positive += int((truth_union & predicted_union).sum())
            false_positive += int((~truth_union & predicted_union).sum())
            false_negative += int((truth_union & ~predicted_union).sum())

            for mask in masks:
                truth_count += 1
                if (mask & predicted_union).sum() / max(mask.sum(), 1) >= 0.5:
                    found_truth += 1

            for mask in selected:
                predicted_count += 1
                if (mask & truth_union).sum() / max(mask.sum(), 1) >= 0.5:
                    correct_predicted += 1

        rows.append({
            'model': model_name,
            'threshold': threshold,
            'pixel_precision': round(true_positive / max(true_positive + false_positive, 1), 3),
            'pixel_recall': round(true_positive / max(true_positive + false_negative, 1), 3),
            'pixel_iou': round(true_positive / max(true_positive + false_positive + false_negative, 1), 3),
            'shape_recall': round(found_truth / max(truth_count, 1), 3),
            'prediction_precision': round(correct_predicted / max(predicted_count, 1), 3),
            'predictions': predicted_count,
        })

with OUTPUT_FILE.open('w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print('model threshold pixel_precision pixel_recall pixel_iou shape_recall prediction_precision predictions')
for row in rows:
    print(*row.values())
print('Saved to:', OUTPUT_FILE)


## 12. Test-zone evaluation (pixel precision / recall / IoU, shape recall)

In [ ]:
# Packages: pip install -r requirements.txt

import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from ultralytics import YOLO


PROJECT_DIR = REPO_DIR
MODEL_FILE = PROJECT_DIR / 'models/seg_rgb_v1/weights/best.pt'
DATASET_DIR = PROJECT_DIR / 'yolo_dataset/full_rgb'
OUTPUT_JSON = PROJECT_DIR / 'outputs/test_results_rgb_conf005.json'
OUTPUT_IMAGE = PROJECT_DIR / 'outputs/test_results_rgb_conf005.png'
CONFIDENCE = 0.05
IMAGE_SIZE = 640


def polygon_mask(points):
    mask = Image.new('L', (IMAGE_SIZE, IMAGE_SIZE), 0)
    if len(points) >= 3:
        ImageDraw.Draw(mask).polygon([tuple(point) for point in points], fill=1)
    return np.array(mask, dtype=bool)


def read_truth(label_file):
    polygons = []
    for line in label_file.read_text().splitlines():
        polygons.append(np.array(line.split()[1:], dtype='float32').reshape(-1, 2) * IMAGE_SIZE)
    return polygons


model = YOLO(MODEL_FILE)
image_files = sorted((DATASET_DIR / 'images/test').glob('*.png'))

true_positive = false_positive = false_negative = 0
truth_count = found_truth = predicted_count = correct_predicted = 0
examples = []

for image_file in image_files:
    truth_polygons = read_truth(DATASET_DIR / 'labels/test' / f'{image_file.stem}.txt')
    truth_masks = [polygon_mask(polygon) for polygon in truth_polygons]
    empty = np.zeros((IMAGE_SIZE, IMAGE_SIZE), dtype=bool)
    truth_union = np.any(truth_masks, axis=0) if truth_masks else empty

    result = model.predict(str(image_file), conf=CONFIDENCE, imgsz=IMAGE_SIZE, verbose=False)[0]
    predicted_polygons = list(result.masks.xy) if result.masks is not None else []
    predicted_masks = [polygon_mask(polygon) for polygon in predicted_polygons]
    predicted_masks = [mask for mask in predicted_masks if mask.any()]
    predicted_union = np.any(predicted_masks, axis=0) if predicted_masks else empty

    true_positive += int((truth_union & predicted_union).sum())
    false_positive += int((~truth_union & predicted_union).sum())
    false_negative += int((truth_union & ~predicted_union).sum())

    for mask in truth_masks:
        truth_count += 1
        if (mask & predicted_union).sum() / max(mask.sum(), 1) >= 0.5:
            found_truth += 1

    for mask in predicted_masks:
        predicted_count += 1
        if (mask & truth_union).sum() / max(mask.sum(), 1) >= 0.5:
            correct_predicted += 1

    examples.append((image_file, truth_polygons, predicted_polygons))

results = {
    'model': 'seg_rgb_v1',
    'confidence': CONFIDENCE,
    'test_tiles': len(image_files),
    'pixel_precision': round(true_positive / max(true_positive + false_positive, 1), 3),
    'pixel_recall': round(true_positive / max(true_positive + false_negative, 1), 3),
    'pixel_iou': round(true_positive / max(true_positive + false_positive + false_negative, 1), 3),
    'shapes_found': found_truth,
    'shapes_total': truth_count,
    'shape_recall': round(found_truth / max(truth_count, 1), 3),
    'predictions_correct': correct_predicted,
    'predictions_total': predicted_count,
    'prediction_precision': round(correct_predicted / max(predicted_count, 1), 3),
}

OUTPUT_JSON.write_text(json.dumps(results, indent=2))
for name, value in results.items():
    print(name, ':', value)

indexes = np.linspace(0, len(examples) - 1, min(8, len(examples))).astype(int)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()
for axis in axes:
    axis.set_axis_off()

for axis, index in zip(axes, indexes):
    image_file, truth_polygons, predicted_polygons = examples[index]
    axis.imshow(Image.open(image_file))
    for polygon in truth_polygons:
        closed = np.vstack([polygon, polygon[0]])
        axis.plot(closed[:, 0], closed[:, 1], color='yellow', linewidth=1.5)
    for polygon in predicted_polygons:
        if len(polygon) >= 3:
            closed = np.vstack([polygon, polygon[0]])
            axis.plot(closed[:, 0], closed[:, 1], color='cyan', linewidth=1)
    axis.set_title(image_file.stem, fontsize=9)

fig.suptitle('TEST zone_1: yellow = truth, cyan = YOLO')
plt.tight_layout()
plt.savefig(OUTPUT_IMAGE, dpi=130, bbox_inches='tight')
plt.show()
print('Saved to:', OUTPUT_IMAGE)


## 13. Detections v1: YOLO on the pilot area

In [ ]:
# Packages: pip install -r requirements.txt

from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from shapely.geometry import Polygon
from shapely.ops import unary_union
from shapely.strtree import STRtree
from ultralytics import YOLO


PROJECT_DIR = REPO_DIR
MODEL_FILE = PROJECT_DIR / 'models/seg_rgb_v1/weights/best.pt'
IMAGE_FILE = PROJECT_DIR / 'data/processed/rgb_8bit.tif'
OUTPUT_DIR = PROJECT_DIR / 'outputs/detections_v1'
LOCAL_DIR = (TEMP_DIR / 'detections_v1')

CONFIDENCE = 0.05
TILE_SIZE = 640
STEP = 320
MIN_AREA_KM2 = 0.5
MERGE_OVERLAP = 0.5

LOCAL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def tile_starts(size):
    starts = list(range(0, size - TILE_SIZE + 1, STEP))
    if starts[-1] != size - TILE_SIZE:
        starts.append(size - TILE_SIZE)
    return starts


def largest_polygon(geometry):
    geometry = geometry.buffer(0)
    if geometry.is_empty:
        return None
    if geometry.geom_type == 'Polygon':
        return geometry
    parts = [part for part in getattr(geometry, 'geoms', []) if part.geom_type == 'Polygon']
    return max(parts, key=lambda part: part.area) if parts else None


model = YOLO(MODEL_FILE)
raw_shapes = []
raw_scores = []

with rasterio.open(IMAGE_FILE) as src:
    crs = src.crs
    transform = src.transform
    bounds = src.bounds
    rows = tile_starts(src.height)
    cols = tile_starts(src.width)
    print('Tiles to process:', len(rows) * len(cols))

    for row in rows:
        for col in cols:
            rgb = src.read(window=Window(col, row, TILE_SIZE, TILE_SIZE))
            if not rgb.any():
                continue
            bgr = np.moveaxis(rgb, 0, -1)[:, :, ::-1].copy()
            result = model.predict(bgr, conf=CONFIDENCE, imgsz=TILE_SIZE, verbose=False)[0]
            if result.masks is None:
                continue
            for points, score in zip(result.masks.xy, result.boxes.conf.cpu().numpy()):
                if len(points) < 3:
                    continue
                coords = [
                    (transform.c + (col + x) * transform.a, transform.f + (row + y) * transform.e)
                    for x, y in points
                ]
                polygon = largest_polygon(Polygon(coords))
                if polygon is not None:
                    raw_shapes.append(polygon)
                    raw_scores.append(float(score))

raw = gpd.GeoDataFrame({'confidence': raw_scores}, geometry=raw_shapes, crs=crs)
print('Raw detections:', len(raw))

tree = STRtree(raw_shapes)
left, right = tree.query(raw_shapes, predicate='intersects')
first_ids = []
second_ids = []
for i, j in zip(left, right):
    if i >= j:
        continue
    shared = raw_shapes[i].intersection(raw_shapes[j]).area
    if shared / min(raw_shapes[i].area, raw_shapes[j].area) >= MERGE_OVERLAP:
        first_ids.append(i)
        second_ids.append(j)

count = len(raw_shapes)
graph = coo_matrix((np.ones(len(first_ids)), (first_ids, second_ids)), shape=(count, count))
group_count, groups = connected_components(graph, directed=False)
raw['group_id'] = groups

records = []
for group_id, group in raw.groupby('group_id'):
    merged = largest_polygon(unary_union(list(group.geometry)).buffer(30).buffer(-30))
    if merged is None:
        continue
    area_km2 = merged.area / 1_000_000
    if area_km2 < MIN_AREA_KM2:
        continue
    rectangle = merged.minimum_rotated_rectangle
    x, y = rectangle.exterior.coords.xy
    length_km = max(np.hypot(x[1] - x[0], y[1] - y[0]), np.hypot(x[2] - x[1], y[2] - y[1])) / 1000
    records.append({
        'max_confidence': round(float(group['confidence'].max()), 3),
        'detections_merged': len(group),
        'area_km2': round(area_km2, 2),
        'est_length_km': round(length_km, 2),
        'mean_width_km': round(area_km2 / length_km, 2),
        'geometry': merged,
    })

detections = gpd.GeoDataFrame(records, crs=crs)
detections = detections.sort_values('max_confidence', ascending=False).reset_index(drop=True)
detections.insert(0, 'detection_id', [f'D{i + 1:04d}' for i in range(len(detections))])

centers = gpd.GeoSeries(detections.geometry.representative_point(), crs=crs).to_crs(4326)
detections['center_lon'] = centers.x.round(6)
detections['center_lat'] = centers.y.round(6)

raw.to_file(LOCAL_DIR / 'raw_yolo_detections.gpkg', driver='GPKG')
detections.to_file(LOCAL_DIR / 'canyon_detections.gpkg', driver='GPKG')
detections.to_crs(4326).to_file(LOCAL_DIR / 'canyon_detections.geojson', driver='GeoJSON')
detections.drop(columns='geometry').to_csv(LOCAL_DIR / 'canyon_detections.csv', index=False)

fig, axis = plt.subplots(figsize=(13, 11))
with rasterio.open(IMAGE_FILE) as src:
    preview = src.read(out_shape=(3, src.height // 5, src.width // 5))
axis.imshow(np.moveaxis(preview, 0, -1), extent=[bounds.left, bounds.right, bounds.bottom, bounds.top])
detections.boundary.plot(ax=axis, color='cyan', linewidth=1)
axis.set_title('YOLO detections after merging')
plt.savefig(LOCAL_DIR / 'detections_map.png', dpi=140, bbox_inches='tight')
plt.show()

for file in LOCAL_DIR.iterdir():
    shutil.copy(file, OUTPUT_DIR / file.name)

print('Final detections:', len(detections))
print('Area min/mean/max km2:', detections.area_km2.min(), round(detections.area_km2.mean(), 2), detections.area_km2.max())
print('All shapes valid:', bool(detections.geometry.is_valid.all()))
print('Latitude range:', detections.center_lat.min(), detections.center_lat.max())
print('Longitude range:', detections.center_lon.min(), detections.center_lon.max())
print(detections.drop(columns='geometry').sort_values('area_km2', ascending=False).head(10).to_string(index=False))
print('Saved to:', OUTPUT_DIR)


## 14. Detections v2: YOLO votes + DEM watershed separation into individual canyons

In [ ]:
# Packages: pip install -r requirements.txt

from pathlib import Path
import shutil
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from rasterio.enums import MergeAlg
from rasterio.features import rasterize, shapes
from scipy.ndimage import gaussian_filter, uniform_filter
from shapely.geometry import shape
from skimage.measure import label
from skimage.morphology import remove_small_objects
from skimage.segmentation import watershed


PROJECT_DIR = REPO_DIR
RAW_FILE = PROJECT_DIR / 'outputs/detections_v1/raw_yolo_detections.gpkg'
DEM_FILE = PROJECT_DIR / 'data/raw/dem_10m_study_area.tif'
IMAGE_FILE = PROJECT_DIR / 'data/processed/rgb_8bit.tif'
TRUTH_FILE = PROJECT_DIR / 'annotations/canyons_ground_truth_v8.gpkg'
OUTPUT_DIR = PROJECT_DIR / 'outputs/detections_v2'
LOCAL_DIR = (TEMP_DIR / 'detections_v2')

MIN_VOTES = 2
STOP_ELEVATION_M = 1000
THRESHOLDS = [-50, -90, -130]
MIN_VALLEY_PIXELS = 5000
MIN_AREA_KM2 = 1
MAX_AREA_KM2 = 25
PIXEL_AREA_KM2 = 0.0001

LOCAL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def largest_polygon(geometry):
    geometry = geometry.buffer(0)
    if geometry.is_empty:
        return None
    if geometry.geom_type == 'Polygon':
        return geometry
    parts = [part for part in getattr(geometry, 'geoms', []) if part.geom_type == 'Polygon']
    return max(parts, key=lambda part: part.area) if parts else None


raw = gpd.read_file(RAW_FILE)
truth = gpd.read_file(TRUTH_FILE)

with rasterio.open(DEM_FILE) as src:
    dem = src.read(1)
    transform = src.transform
    crs = src.crs
    bounds = src.bounds

votes = rasterize(
    [(geometry, 1) for geometry in raw.geometry],
    out_shape=dem.shape,
    transform=transform,
    fill=0,
    dtype='uint16',
    merge_alg=MergeAlg.add,
)

land = (votes >= MIN_VOTES) & (dem >= STOP_ELEVATION_M)
smooth = gaussian_filter(dem, sigma=8)
position = smooth - uniform_filter(smooth, size=101, mode='nearest')

remaining = land.copy()
accepted_regions = np.zeros(dem.shape, dtype='int32')
next_id = 1

for threshold in THRESHOLDS:
    valleys = remove_small_objects((position <= threshold) & remaining, min_size=MIN_VALLEY_PIXELS)
    markers = label(valleys, connectivity=2)
    if markers.max() == 0:
        continue
    regions = watershed(smooth, markers=markers, mask=remaining)
    area = np.bincount(regions.ravel()) * PIXEL_AREA_KM2
    keep = (area >= MIN_AREA_KM2) & (area <= MAX_AREA_KM2)
    keep[0] = False
    for region_id in np.flatnonzero(keep):
        selected = regions == region_id
        accepted_regions[selected] = next_id
        remaining[selected] = False
        next_id += 1

valleys = remove_small_objects((position <= THRESHOLDS[-1]) & remaining, min_size=MIN_VALLEY_PIXELS)
markers = label(valleys, connectivity=2)
large_ids = set()
if markers.max() > 0:
    regions = watershed(smooth, markers=markers, mask=remaining)
    area = np.bincount(regions.ravel()) * PIXEL_AREA_KM2
    for region_id in np.flatnonzero(area >= MIN_AREA_KM2):
        if region_id == 0:
            continue
        accepted_regions[regions == region_id] = next_id
        large_ids.add(next_id)
        next_id += 1

records = []
for geometry, value in shapes(accepted_regions, mask=accepted_regions > 0, transform=transform):
    polygon = largest_polygon(shape(geometry).buffer(-30).buffer(30))
    if polygon is None:
        continue
    polygon = largest_polygon(polygon.simplify(20))
    if polygon is not None and polygon.area / 1_000_000 >= MIN_AREA_KM2:
        records.append({'region': int(value), 'large_region': int(value) in large_ids, 'geometry': polygon})

detections = gpd.GeoDataFrame(records, crs=crs)
detections = detections.sort_values('region').reset_index(drop=True)

clean_shapes = []
used = None
for geometry in detections.geometry:
    cleaned = geometry if used is None else largest_polygon(geometry.difference(used))
    clean_shapes.append(cleaned)
    if cleaned is not None:
        used = cleaned if used is None else used.union(cleaned)

detections['geometry'] = clean_shapes
detections = detections[detections.geometry.notna()].reset_index(drop=True)
detections = detections[detections.geometry.area >= MIN_AREA_KM2 * 1_000_000].reset_index(drop=True)

raw['raw_id'] = raw.index
pairs = gpd.sjoin(raw[['raw_id', 'confidence', 'geometry']], detections[['geometry']], predicate='intersects')
max_conf = pairs.groupby('index_right')['confidence'].max()
mean_conf = pairs.groupby('index_right')['confidence'].mean()

length_km = []
for geometry in detections.geometry:
    x, y = geometry.minimum_rotated_rectangle.exterior.coords.xy
    length_km.append(max(np.hypot(x[1] - x[0], y[1] - y[0]), np.hypot(x[2] - x[1], y[2] - y[1])) / 1000)

detections['area_km2'] = (detections.geometry.area / 1_000_000).round(2)
detections['est_length_km'] = np.round(length_km, 2)
detections['mean_width_km'] = (detections['area_km2'] / detections['est_length_km']).round(2)
detections['max_confidence'] = detections.index.map(max_conf).astype(float).round(3)
detections['mean_confidence'] = detections.index.map(mean_conf).astype(float).round(3)

detections = detections.sort_values('mean_confidence', ascending=False).reset_index(drop=True)
detections.insert(0, 'detection_id', [f'D{i + 1:04d}' for i in range(len(detections))])
centers = gpd.GeoSeries(detections.geometry.representative_point(), crs=crs).to_crs(4326)
detections['center_lon'] = centers.x.round(6)
detections['center_lat'] = centers.y.round(6)
detections = detections.drop(columns='region')

detections.to_file(LOCAL_DIR / 'canyon_detections.gpkg', driver='GPKG')
detections.to_crs(4326).to_file(LOCAL_DIR / 'canyon_detections.geojson', driver='GeoJSON')
detections.drop(columns='geometry').to_csv(LOCAL_DIR / 'canyon_detections.csv', index=False)

manual_names = {f'C{i:03d}' for i in range(1, 11)}
manual_matches = 0
auto_matches = 0
for _, canyon in truth.iterrows():
    possible = detections[detections.intersects(canyon.geometry)]
    best = 0
    for geometry in possible.geometry:
        shared = canyon.geometry.intersection(geometry).area
        best = max(best, shared / canyon.geometry.union(geometry).area)
    if best >= 0.5:
        if canyon.canyon_name in manual_names:
            manual_matches += 1
        else:
            auto_matches += 1

with rasterio.open(IMAGE_FILE) as src:
    preview = src.read(out_shape=(3, src.height // 5, src.width // 5))

fig, axis = plt.subplots(figsize=(14, 11))
axis.imshow(np.moveaxis(preview, 0, -1), extent=[bounds.left, bounds.right, bounds.bottom, bounds.top])
detections[~detections.large_region].boundary.plot(ax=axis, color='cyan', linewidth=0.9)
if detections.large_region.any():
    detections[detections.large_region].boundary.plot(ax=axis, color='orange', linewidth=1.3)
axis.set_xlim(bounds.left, bounds.right)
axis.set_ylim(bounds.bottom, bounds.top)
axis.set_title('Detections v2: cyan = separated, orange = large regions to review')
plt.savefig(LOCAL_DIR / 'detections_map.png', dpi=140, bbox_inches='tight')
plt.show()

for file in LOCAL_DIR.iterdir():
    shutil.copy(file, OUTPUT_DIR / file.name)

print('Final detections:', len(detections))
print('Large regions to review:', int(detections.large_region.sum()))
print('Area min/mean/max km2:', detections.area_km2.min(), round(detections.area_km2.mean(), 2), detections.area_km2.max())
print('All shapes valid:', bool(detections.geometry.is_valid.all()))
print('Manual canyons matched at IoU 0.5:', manual_matches, 'of 10')
print('Automatic reference shapes matched at IoU 0.5:', auto_matches, 'of', len(truth) - 10)
print(detections.drop(columns=['geometry', 'large_region']).sort_values('area_km2', ascending=False).head(10).to_string(index=False))
print('Saved to:', OUTPUT_DIR)


## 15. Extra reference areas, part 1 (Marble Canyon, western Grand Canyon)

In [ ]:
# Packages: pip install -r requirements.txt

import importlib
import shutil
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.features import shapes
from scipy.ndimage import binary_fill_holes, gaussian_filter, maximum_filter, minimum_filter, uniform_filter
from shapely.geometry import shape
from skimage.measure import label
from skimage.morphology import remove_small_objects
from skimage.segmentation import watershed


PROJECT_DIR = REPO_DIR
sys.path.insert(0, str(REPO_DIR))
import detector
importlib.reload(detector)

OUTPUT_DIR = PROJECT_DIR / 'annotations/expansion'
LOCAL_DIR = (TEMP_DIR / 'expansion_zones')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(exist_ok=True)

ZONES = {
    'marble_train': (-111.95, 36.30, -111.75, 36.45),
    'west_a_train': (-113.50, 35.75, -113.30, 35.91),
    'west_b_test': (-113.85, 35.90, -113.65, 36.06),
}
DATE_RANGE = '2026-05-15/2026-08-31'
THRESHOLDS = [-50, -90, -130]


def largest_polygon(geometry):
    geometry = geometry.buffer(0)
    if geometry.is_empty:
        return None
    if geometry.geom_type == 'Polygon':
        return geometry
    parts = [part for part in getattr(geometry, 'geoms', []) if part.geom_type == 'Polygon']
    return max(parts, key=lambda part: part.area) if parts else None


def split_regions(mask, position, smooth):
    remaining = mask.copy()
    output = np.zeros(mask.shape, dtype='int32')
    next_id = 1
    for threshold in THRESHOLDS:
        valleys = remove_small_objects((position <= threshold) & remaining, min_size=5000)
        markers = label(valleys, connectivity=2)
        if markers.max() == 0:
            continue
        regions = watershed(smooth, markers=markers, mask=remaining)
        area = np.bincount(regions.ravel()) * 0.0001
        keep = (area >= 2) & (area <= 25)
        keep[0] = False
        for region_id in np.flatnonzero(keep):
            selected = regions == region_id
            output[selected] = next_id
            remaining[selected] = False
            next_id += 1
    return output


for zone_name, bbox in ZONES.items():
    print('Starting:', zone_name)
    transform, width, height, area_km2 = detector.prepare_grid(*bbox)
    catalog = detector.open_catalog()
    scene_date, items = detector.choose_scenes(catalog, list(bbox), DATE_RANGE)
    rgb = detector.load_rgb(items, transform, width, height)
    dem = detector.load_dem(catalog, list(bbox), transform, width, height)

    stop_m = detector.stop_elevation((bbox[0] + bbox[2]) / 2)
    relief = maximum_filter(dem, size=101) - minimum_filter(dem, size=101)
    mask = binary_fill_holes(relief >= 250) & (dem >= stop_m)
    mask &= (maximum_filter(dem, size=101) - dem) >= 60

    smooth = gaussian_filter(dem, sigma=8)
    position = smooth - uniform_filter(smooth, size=101, mode='nearest')
    regions = split_regions(mask, position, smooth)

    records = []
    for geometry, value in shapes(regions, mask=regions > 0, transform=transform):
        polygon = largest_polygon(shape(geometry).buffer(-40).buffer(40))
        polygon = largest_polygon(polygon.simplify(30)) if polygon is not None else None
        if polygon is not None and polygon.area >= 1_500_000:
            records.append({'area_km2': round(polygon.area / 1_000_000, 2), 'geometry': polygon})

    prefix = zone_name[0].upper() + zone_name.split('_')[1][0].upper()
    candidates = gpd.GeoDataFrame(records, geometry='geometry', crs=detector.TARGET_CRS)
    candidates.insert(0, 'cand_id', [f'{prefix}{i + 1:03d}' for i in range(len(candidates))])

    local_gpkg = LOCAL_DIR / f'{zone_name}_candidates.gpkg'
    local_tif = LOCAL_DIR / f'{zone_name}_rgb.tif'
    local_png = LOCAL_DIR / f'{zone_name}_review.png'
    candidates.to_file(local_gpkg, driver='GPKG')

    with rasterio.open(local_tif, 'w', driver='GTiff', height=height, width=width, count=3, dtype='uint8',
                       crs=detector.TARGET_CRS, transform=transform, nodata=0, compress='deflate', tiled=True) as dst:
        dst.write(rgb)

    scaled = np.clip(position / 200, -1, 1)
    ridge = np.stack([
        1 + np.minimum(scaled, 0),
        1 - np.abs(scaled),
        1 - np.maximum(scaled, 0),
    ], axis=-1)

    left, top = transform.c, transform.f
    extent = [left, left + width * 10, top - height * 10, top]
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    axes[0].imshow(np.moveaxis(rgb, 0, -1), extent=extent)
    axes[0].set_title(f'{zone_name}: Sentinel-2 ({scene_date})')
    axes[1].imshow(ridge, extent=extent)
    axes[1].set_title('Blue = valleys, red = ridges')

    for axis in axes:
        candidates.boundary.plot(ax=axis, color='black', linewidth=1.3)
        for _, item in candidates.iterrows():
            point = item.geometry.representative_point()
            axis.text(point.x, point.y, item.cand_id, fontsize=8, fontweight='bold', ha='center',
                      bbox={'facecolor': 'white', 'alpha': 0.75, 'pad': 1})
        axis.set_xlim(extent[0], extent[1])
        axis.set_ylim(extent[2], extent[3])

    plt.tight_layout()
    plt.savefig(local_png, dpi=130, bbox_inches='tight')
    plt.show()

    for file in [local_gpkg, local_tif, local_png]:
        shutil.copy(file, OUTPUT_DIR / file.name)

    print(zone_name, '| date:', scene_date, '| area km2:', round(area_km2, 1), '| candidates:', len(candidates))

print('Finished')


## 16. Extra reference areas, part 2 (Marble Canyon v2, Lava Falls test area)

In [ ]:
# Packages: pip install -r requirements.txt

import importlib
import shutil
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.features import shapes
from scipy.ndimage import binary_fill_holes, gaussian_filter, maximum_filter, minimum_filter, uniform_filter
from shapely.geometry import Polygon, shape
from skimage.measure import label
from skimage.morphology import remove_small_objects
from skimage.segmentation import watershed


PROJECT_DIR = REPO_DIR
sys.path.insert(0, str(REPO_DIR))
import detector
importlib.reload(detector)

OUTPUT_DIR = PROJECT_DIR / 'annotations/expansion'
LOCAL_DIR = (TEMP_DIR / 'expansion_zones_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(exist_ok=True)

ZONES = {
    'marble_v2_train': {'bbox': (-111.95, 36.30, -111.75, 36.45), 'stop_m': 1250, 'prefix': 'MR'},
    'lava_test': {'bbox': (-113.25, 36.10, -113.05, 36.26), 'stop_m': 700, 'prefix': 'LT'},
}
DATE_RANGE = '2026-05-15/2026-08-31'
THRESHOLDS = [-50, -90, -130]


def largest_polygon(geometry):
    geometry = geometry.buffer(0)
    if geometry.is_empty:
        return None
    if geometry.geom_type == 'Polygon':
        return geometry
    parts = [part for part in getattr(geometry, 'geoms', []) if part.geom_type == 'Polygon']
    return max(parts, key=lambda part: part.area) if parts else None


def split_regions(mask, position, smooth):
    remaining = mask.copy()
    output = np.zeros(mask.shape, dtype='int32')
    next_id = 1
    for threshold in THRESHOLDS:
        valleys = remove_small_objects((position <= threshold) & remaining, min_size=5000)
        markers = label(valleys, connectivity=2)
        if markers.max() == 0:
            continue
        regions = watershed(smooth, markers=markers, mask=remaining)
        area = np.bincount(regions.ravel()) * 0.0001
        keep = (area >= 2) & (area <= 25)
        keep[0] = False
        for region_id in np.flatnonzero(keep):
            selected = regions == region_id
            output[selected] = next_id
            remaining[selected] = False
            next_id += 1
    return output


for zone_name, zone in ZONES.items():
    print('Starting:', zone_name)
    bbox = zone['bbox']
    transform, width, height, area_km2 = detector.prepare_grid(*bbox)
    catalog = detector.open_catalog()
    scene_date, items = detector.choose_scenes(catalog, list(bbox), DATE_RANGE)
    rgb = detector.load_rgb(items, transform, width, height)
    dem = detector.load_dem(catalog, list(bbox), transform, width, height)

    relief = maximum_filter(dem, size=101) - minimum_filter(dem, size=101)
    mask = binary_fill_holes(relief >= 250) & (dem >= zone['stop_m'])
    mask &= (maximum_filter(dem, size=101) - dem) >= 60

    smooth = gaussian_filter(dem, sigma=8)
    position = smooth - uniform_filter(smooth, size=101, mode='nearest')
    regions = split_regions(mask, position, smooth)

    records = []
    for geometry, value in shapes(regions, mask=regions > 0, transform=transform):
        polygon = largest_polygon(shape(geometry).buffer(-40).buffer(40))
        if polygon is None:
            continue
        polygon = largest_polygon(Polygon(polygon.exterior).simplify(30))
        if polygon is not None and polygon.area >= 1_500_000:
            records.append({'area_km2': round(polygon.area / 1_000_000, 2), 'geometry': polygon})

    candidates = gpd.GeoDataFrame(records, geometry='geometry', crs=detector.TARGET_CRS)
    candidates.insert(0, 'cand_id', [f"{zone['prefix']}{i + 1:03d}" for i in range(len(candidates))])

    local_gpkg = LOCAL_DIR / f'{zone_name}_candidates.gpkg'
    local_tif = LOCAL_DIR / f'{zone_name}_rgb.tif'
    local_png = LOCAL_DIR / f'{zone_name}_review.png'
    candidates.to_file(local_gpkg, driver='GPKG')

    with rasterio.open(local_tif, 'w', driver='GTiff', height=height, width=width, count=3, dtype='uint8',
                       crs=detector.TARGET_CRS, transform=transform, nodata=0, compress='deflate', tiled=True) as dst:
        dst.write(rgb)

    scaled = np.clip(position / 200, -1, 1)
    ridge = np.stack([
        1 + np.minimum(scaled, 0),
        1 - np.abs(scaled),
        1 - np.maximum(scaled, 0),
    ], axis=-1)

    left, top = transform.c, transform.f
    extent = [left, left + width * 10, top - height * 10, top]
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    axes[0].imshow(np.moveaxis(rgb, 0, -1), extent=extent)
    axes[0].set_title(f'{zone_name}: Sentinel-2 ({scene_date})')
    axes[1].imshow(ridge, extent=extent)
    axes[1].set_title('Blue = valleys, red = ridges')

    for axis in axes:
        if len(candidates):
            candidates.boundary.plot(ax=axis, color='black', linewidth=1.3)
        for _, item in candidates.iterrows():
            point = item.geometry.representative_point()
            axis.text(point.x, point.y, item.cand_id, fontsize=8, fontweight='bold', ha='center',
                      bbox={'facecolor': 'white', 'alpha': 0.75, 'pad': 1})
        axis.set_xlim(extent[0], extent[1])
        axis.set_ylim(extent[2], extent[3])

    plt.tight_layout()
    plt.savefig(local_png, dpi=130, bbox_inches='tight')
    plt.show()

    for file in [local_gpkg, local_tif, local_png]:
        shutil.copy(file, OUTPUT_DIR / file.name)

    print(zone_name, '| date:', scene_date, '| area km2:', round(area_km2, 1), '| candidates:', len(candidates))

print('Finished')


## 17. Train and evaluate model v2 (adds the extra areas) — writes models/seg_rgb_v2 and outputs/model_v2_results.csv

In [ ]:
# Packages: pip install -r requirements.txt

import csv
import random
import shutil
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import rasterio
import torch
from PIL import Image, ImageDraw
from rasterio.features import rasterize
from rasterio.windows import Window, bounds as window_bounds
from scipy.ndimage import binary_fill_holes, maximum_filter, minimum_filter
from shapely.geometry import Polygon, box
from shapely.ops import unary_union
from ultralytics import YOLO


if not torch.cuda.is_available():
    raise RuntimeError('GPU is not active. Select T4 GPU, then run again.')

PROJECT_DIR = REPO_DIR
sys.path.insert(0, str(REPO_DIR))
import detector

EXPANSION_DIR = PROJECT_DIR / 'annotations/expansion'
BASE_DATASET = PROJECT_DIR / 'yolo_dataset/full_rgb'
DATASET_DIR = (TEMP_DIR / 'datasets/rgb_v2')
RUNS_DIR = (TEMP_DIR / 'local_runs')
MODEL_DIR = PROJECT_DIR / 'models/seg_rgb_v2'
RESULT_FILE = PROJECT_DIR / 'outputs/model_v2_results.csv'

ZONES = {
    'west_b_test': {
        'bbox': (-113.85, 35.90, -113.65, 36.06), 'stop_m': 700, 'split': 'train',
        'accept': [f'WB{i:03d}' for i in range(1, 16) if i != 7],
    },
    'marble_v2_train': {
        'bbox': (-111.95, 36.30, -111.75, 36.45), 'stop_m': 1250, 'split': 'train',
        'accept': ['MR001', 'MR004', 'MR005', 'MR006', 'MR007'],
    },
    'lava_test': {
        'bbox': (-113.25, 36.10, -113.05, 36.26), 'stop_m': 700, 'split': 'test_lava',
        'accept': ['LT001', 'LT002', 'LT004', 'LT005'],
    },
}
TILE_SIZE = 640
STEP = 160
MIN_LABEL_AREA_M2 = 100000
MAX_UNLABELED_SHARE = 0.25
EMPTY_TILE_SHARE = 0.2
THRESHOLDS = [0.05, 0.10, 0.15, 0.25]
random.seed(42)


def polygon_parts(geometry):
    if geometry.is_empty:
        return []
    if geometry.geom_type == 'Polygon':
        return [geometry]
    return [part for part in getattr(geometry, 'geoms', []) if part.geom_type == 'Polygon']


def largest_polygon(geometry):
    parts = polygon_parts(geometry.buffer(0))
    return max(parts, key=lambda part: part.area) if parts else None


def label_lines(shapes, tile_box, left, top, tile_width):
    lines = []
    for geometry in shapes:
        if not geometry.intersects(tile_box):
            continue
        for part in polygon_parts(geometry.intersection(tile_box)):
            if part.area < MIN_LABEL_AREA_M2:
                continue
            values = ['0']
            for x, y in list(part.exterior.coords)[:-1]:
                values.append(f'{np.clip((x - left) / tile_width, 0, 1):.6f}')
                values.append(f'{np.clip((top - y) / tile_width, 0, 1):.6f}')
            lines.append(' '.join(values))
    return lines


if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
shutil.copytree(BASE_DATASET, DATASET_DIR)
(DATASET_DIR / 'images/test_lava').mkdir(parents=True)
(DATASET_DIR / 'labels/test_lava').mkdir(parents=True)

catalog = detector.open_catalog()

for zone_name, zone in ZONES.items():
    print('Adding:', zone_name)
    candidates = gpd.read_file(EXPANSION_DIR / f'{zone_name}_candidates.gpkg')
    chosen = candidates[candidates['cand_id'].isin(zone['accept'])]
    if len(chosen) != len(zone['accept']):
        raise ValueError(f'Some accepted IDs are missing in {zone_name}')

    clean_shapes = []
    used = None
    for geometry in chosen.geometry:
        current = largest_polygon(Polygon(largest_polygon(geometry).exterior))
        if used is not None:
            current = largest_polygon(current.difference(used))
        if current is not None:
            clean_shapes.append(current)
            used = current if used is None else unary_union([used, current])

    with rasterio.open(EXPANSION_DIR / f'{zone_name}_rgb.tif') as src:
        rgb = src.read()
        transform = src.transform
        height, width = src.height, src.width

    dem = detector.load_dem(catalog, list(zone['bbox']), transform, width, height)
    relief = maximum_filter(dem, size=101) - minimum_filter(dem, size=101)
    canyon_like = binary_fill_holes(relief >= 250) & (dem >= zone['stop_m'])
    canyon_like &= (maximum_filter(dem, size=101) - dem) >= 60
    labeled = rasterize([(geometry, 1) for geometry in clean_shapes], out_shape=dem.shape,
                        transform=transform, fill=0, dtype='uint8').astype(bool)
    unlabeled = canyon_like & ~labeled

    labeled_tiles = []
    empty_tiles = []
    tile_width = TILE_SIZE * 10

    for row in range(0, height - TILE_SIZE + 1, STEP):
        for col in range(0, width - TILE_SIZE + 1, STEP):
            if unlabeled[row:row + TILE_SIZE, col:col + TILE_SIZE].mean() > MAX_UNLABELED_SHARE:
                continue
            left, bottom, right, top = window_bounds(Window(col, row, TILE_SIZE, TILE_SIZE), transform)
            lines = label_lines(clean_shapes, box(left, bottom, right, top), left, top, tile_width)
            (labeled_tiles if lines else empty_tiles).append((row, col, lines))

    keep_empty = min(len(empty_tiles), int(len(labeled_tiles) * EMPTY_TILE_SHARE))
    tiles = labeled_tiles + random.sample(empty_tiles, keep_empty)

    for row, col, lines in tiles:
        name = f'{zone_name}_r{row:05d}_c{col:05d}'
        image = np.moveaxis(rgb[:, row:row + TILE_SIZE, col:col + TILE_SIZE], 0, -1)
        Image.fromarray(image).save(DATASET_DIR / 'images' / zone['split'] / f'{name}.png')
        text = '\n'.join(lines)
        (DATASET_DIR / 'labels' / zone['split'] / f'{name}.txt').write_text(text + ('\n' if text else ''))

    print(zone_name, '| shapes:', len(clean_shapes), '| labeled tiles:', len(labeled_tiles), '| empty tiles:', keep_empty)

data_file = DATASET_DIR / 'data.yaml'
data_file.write_text(
    f'path: {DATASET_DIR}\n'
    'train: images/train\n'
    'val: images/val\n'
    'test: images/test\n'
    'names:\n'
    '  0: canyon\n'
)

for split in ['train', 'val', 'test', 'test_lava']:
    print(split, 'tiles:', len(list((DATASET_DIR / 'images' / split).glob('*.png'))))

model = YOLO('yolo26s-seg.pt')
model.train(
    data=str(data_file),
    epochs=100,
    patience=30,
    imgsz=640,
    batch=8,
    seed=42,
    workers=2,
    project=str(RUNS_DIR),
    name='seg_rgb_v2',
    exist_ok=True,
    verbose=False,
)

best_file = Path(model.trainer.best)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
shutil.copyfile(best_file, MODEL_DIR / 'best.pt')
shutil.copyfile(best_file.parent.parent / 'results.csv', MODEL_DIR / 'results.csv')


def polygon_mask(points):
    mask = Image.new('L', (TILE_SIZE, TILE_SIZE), 0)
    if len(points) >= 3:
        ImageDraw.Draw(mask).polygon([tuple(point) for point in points], fill=1)
    return np.array(mask, dtype=bool)


def truth_masks(label_file):
    masks = []
    for line in label_file.read_text().splitlines():
        points = np.array(line.split()[1:], dtype='float32').reshape(-1, 2) * TILE_SIZE
        masks.append(polygon_mask(points))
    return masks


def evaluate(trained_model, split, threshold):
    tp = fp = fn = truth_count = found = 0
    empty = np.zeros((TILE_SIZE, TILE_SIZE), dtype=bool)
    for image_file in sorted((DATASET_DIR / 'images' / split).glob('*.png')):
        masks = truth_masks(DATASET_DIR / 'labels' / split / f'{image_file.stem}.txt')
        result = trained_model.predict(str(image_file), conf=threshold, imgsz=TILE_SIZE, verbose=False)[0]
        predicted = [polygon_mask(points) for points in result.masks.xy] if result.masks is not None else []
        truth_union = np.any(masks, axis=0) if masks else empty
        predicted_union = np.any(predicted, axis=0) if predicted else empty
        tp += int((truth_union & predicted_union).sum())
        fp += int((~truth_union & predicted_union).sum())
        fn += int((truth_union & ~predicted_union).sum())
        for mask in masks:
            truth_count += 1
            if (mask & predicted_union).sum() / max(mask.sum(), 1) >= 0.5:
                found += 1
    return {
        'pixel_precision': round(tp / max(tp + fp, 1), 3),
        'pixel_recall': round(tp / max(tp + fn, 1), 3),
        'pixel_iou': round(tp / max(tp + fp + fn, 1), 3),
        'shape_recall': round(found / max(truth_count, 1), 3),
    }


old_model = YOLO(PROJECT_DIR / 'models/seg_rgb_v1/weights/best.pt')
new_model = YOLO(MODEL_DIR / 'best.pt')
rows = []

for threshold in THRESHOLDS:
    rows.append({'model': 'v2', 'split': 'val', 'threshold': threshold, **evaluate(new_model, 'val', threshold)})

validation_rows = [row for row in rows if row['split'] == 'val']
best_threshold = max(validation_rows, key=lambda row: row['pixel_iou'])['threshold']

for split in ['test', 'test_lava']:
    rows.append({'model': 'v1', 'split': split, 'threshold': 0.05, **evaluate(old_model, split, 0.05)})
    rows.append({'model': 'v2', 'split': split, 'threshold': best_threshold, **evaluate(new_model, split, best_threshold)})

with RESULT_FILE.open('w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print('Training finished')
print('Best validation threshold:', best_threshold)
print('model split threshold pixel_precision pixel_recall pixel_iou shape_recall')
for row in rows:
    print(*row.values())
print('Saved to:', RESULT_FILE)


## 18. Run the v2 detector over the whole Grand Canyon in 157 resumable blocks

In [ ]:
# Packages: pip install -r requirements.txt

import importlib
import json
import shutil
import sys
import time
from pathlib import Path

import numpy as np
from shapely.geometry import LineString, box


PROJECT_DIR = REPO_DIR
APP_DIR = PROJECT_DIR / 'app'
BLOCKS_DIR = PROJECT_DIR / 'outputs/expansion_v3/blocks'
LOCAL_DIR = (TEMP_DIR / 'expansion_blocks_v3')
BLOCKS_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(exist_ok=True)

# detector.py in this repository already uses model v2 and the updated stop elevation.

sys.path.insert(0, str(REPO_DIR))
import detector
importlib.reload(detector)

WEST, SOUTH, EAST, NORTH = -114.25, 35.45, -111.25, 37.05
CORE_LON = 0.16
CORE_LAT = 0.13
BUFFER_DEG = 0.02
RIVER_BUFFER = 0.40
MAX_MINUTES = 150

river = LineString([
    (-111.60, 36.87), (-111.85, 36.45), (-111.79, 36.19), (-112.09, 36.10),
    (-112.40, 36.20), (-112.63, 36.39), (-112.76, 36.30), (-113.09, 36.19),
    (-113.37, 35.77), (-113.57, 35.90), (-114.00, 36.12),
])
area_near_river = river.buffer(RIVER_BUFFER)

blocks = []
for row, south in enumerate(np.arange(SOUTH + BUFFER_DEG, NORTH - CORE_LAT - BUFFER_DEG, CORE_LAT)):
    for col, west in enumerate(np.arange(WEST + BUFFER_DEG, EAST - CORE_LON - BUFFER_DEG, CORE_LON)):
        core = box(west, south, west + CORE_LON, south + CORE_LAT)
        if core.intersects(area_near_river):
            blocks.append((f'block_r{row:02d}_c{col:02d}', core))

print('Model:', detector.MODEL_FILE)
print('Blocks to process:', len(blocks))

started = time.time()
new_count = 0
errors = {}

for name, core in blocks:
    output_file = BLOCKS_DIR / f'{name}.geojson'
    empty_file = BLOCKS_DIR / f'{name}.empty'
    if output_file.exists() or empty_file.exists():
        continue
    if (time.time() - started) / 60 > MAX_MINUTES:
        print('Time limit reached. Run this cell again to continue.')
        break

    west, south, east, north = core.bounds
    try:
        result = detector.detect_area(
            west - BUFFER_DEG,
            south - BUFFER_DEG,
            east + BUFFER_DEG,
            north + BUFFER_DEG,
            progress=lambda text: None,
        )
    except Exception as error:
        errors[name] = str(error)
        print(name, '| error:', error)
        continue

    detections = result['detections']
    if len(detections) > 0:
        detections = detections.to_crs(4326)
        points = detections.geometry.representative_point()
        detections = detections[points.within(core)].copy()

    if len(detections) == 0:
        empty_file.write_text('no detections')
    else:
        detections['block'] = name
        detections['scene_date'] = result['scene_date']
        local_file = LOCAL_DIR / output_file.name
        detections.to_file(local_file, driver='GeoJSON')
        shutil.copy(local_file, output_file)

    new_count += 1
    finished = len(list(BLOCKS_DIR.glob('*.geojson'))) + len(list(BLOCKS_DIR.glob('*.empty')))
    print(name, '| detections:', len(detections), '| finished:', finished, 'of', len(blocks))

(BLOCKS_DIR.parent / 'errors.json').write_text(json.dumps(errors, indent=2))
finished = len(list(BLOCKS_DIR.glob('*.geojson'))) + len(list(BLOCKS_DIR.glob('*.empty')))
print('New blocks this run:', new_count)
print('Blocks finished:', finished, 'of', len(blocks))
print('Blocks with errors this run:', len(errors))


## 19. Merge the blocks into the final catalogue (outputs/grand_canyon_v2)

In [ ]:
# Packages: pip install -r requirements.txt

from pathlib import Path
import shutil

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import box


PROJECT_DIR = REPO_DIR
BLOCKS_DIR = PROJECT_DIR / 'outputs/expansion_v3/blocks'
OUTPUT_DIR = PROJECT_DIR / 'outputs/grand_canyon_v2'
LOCAL_DIR = (TEMP_DIR / 'grand_canyon_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(exist_ok=True)

UTM_CRS = 'EPSG:32612'
MIN_AREA_KM2 = 1
PILOT_BOX = box(-112.45, 35.95, -111.85, 36.35)

block_files = sorted(BLOCKS_DIR.glob('*.geojson'))
empty_files = sorted(BLOCKS_DIR.glob('*.empty'))
block_names = {file.stem for file in block_files}
empty_names = {file.stem for file in empty_files}
duplicates = block_names & empty_names

print('Unique blocks:', len(block_names | empty_names))
print('Blocks with detections:', len(block_names))
print('Blocks without detections:', len(empty_names - block_names))
print('Blocks saved as both:', sorted(duplicates))

all_rows = pd.concat([gpd.read_file(file) for file in block_files], ignore_index=True)
detections = gpd.GeoDataFrame(all_rows, geometry='geometry', crs=4326).to_crs(UTM_CRS)
print('Shapes before overlap removal:', len(detections))


def largest_polygon(geometry):
    geometry = geometry.buffer(0)
    if geometry.is_empty:
        return None
    if geometry.geom_type == 'Polygon':
        return geometry
    parts = [part for part in getattr(geometry, 'geoms', []) if part.geom_type == 'Polygon']
    return max(parts, key=lambda part: part.area) if parts else None


detections = detections.sort_values('mean_confidence', ascending=False).reset_index(drop=True)
accepted = []
accepted_rows = []

for index, row in detections.iterrows():
    current = largest_polygon(row.geometry)
    if current is None:
        continue
    for previous in accepted:
        if current.intersects(previous):
            current = largest_polygon(current.difference(previous))
            if current is None:
                break
    if current is not None and current.area / 1_000_000 >= MIN_AREA_KM2:
        accepted.append(current)
        accepted_rows.append(index)

final = detections.loc[accepted_rows].copy()
final['geometry'] = accepted
final = final.reset_index(drop=True)

lengths = []
for geometry in final.geometry:
    x, y = geometry.minimum_rotated_rectangle.exterior.coords.xy
    lengths.append(max(np.hypot(x[1] - x[0], y[1] - y[0]), np.hypot(x[2] - x[1], y[2] - y[1])) / 1000)

final['area_km2'] = (final.geometry.area / 1_000_000).round(2)
final['est_length_km'] = np.round(lengths, 2)
final['mean_width_km'] = (final['area_km2'] / final['est_length_km']).round(2)
centers = gpd.GeoSeries(final.geometry.representative_point(), crs=UTM_CRS).to_crs(4326)
final['center_lon'] = centers.x.round(6)
final['center_lat'] = centers.y.round(6)
final['inside_tested_area'] = centers.within(PILOT_BOX).to_numpy()
final['detection_id'] = [f'G{i + 1:04d}' for i in range(len(final))]

columns = [
    'detection_id', 'area_km2', 'est_length_km', 'mean_width_km', 'max_confidence', 'mean_confidence',
    'center_lon', 'center_lat', 'inside_tested_area', 'block', 'scene_date', 'geometry',
]
final = final[columns]
final_wgs84 = final.to_crs(4326)

final.to_file(LOCAL_DIR / 'grand_canyon_detections.gpkg', driver='GPKG')
final_wgs84.to_file(LOCAL_DIR / 'grand_canyon_detections.geojson', driver='GeoJSON')
final.drop(columns='geometry').to_csv(LOCAL_DIR / 'grand_canyon_detections.csv', index=False)

fig, axis = plt.subplots(figsize=(16, 9))
colors = np.where(
    final_wgs84.mean_confidence >= 0.6,
    '#22c55e',
    np.where(final_wgs84.mean_confidence >= 0.3, '#facc15', '#f97316'),
)
final_wgs84.plot(ax=axis, color=colors, edgecolor='black', linewidth=0.2)
gpd.GeoSeries([PILOT_BOX], crs=4326).boundary.plot(ax=axis, color='blue', linewidth=2)
axis.set_title('Grand Canyon detections, model v2. Blue box = original pilot area')
axis.set_xlabel('Longitude')
axis.set_ylabel('Latitude')
axis.set_aspect('equal')
plt.savefig(LOCAL_DIR / 'grand_canyon_map.png', dpi=140, bbox_inches='tight')
plt.show()

for file in LOCAL_DIR.iterdir():
    shutil.copy(file, OUTPUT_DIR / file.name)

print('Final detections:', len(final))
print('Inside original pilot area:', int(final.inside_tested_area.sum()))
print('Outside original pilot area:', int((~final.inside_tested_area).sum()))
print('Total area km2:', round(final.area_km2.sum(), 2))
print('Area min/mean/max km2:', final.area_km2.min(), round(final.area_km2.mean(), 2), final.area_km2.max())
print('Mean confidence:', round(final.mean_confidence.mean(), 3))
print('Confidence 0.3 or more:', int((final.mean_confidence >= 0.3).sum()))
print('All shapes valid:', bool(final.geometry.is_valid.all()))
print('Saved to:', OUTPUT_DIR)


## 20. Attach USGS SGMC geology to every detection

In [ ]:
# Packages: pip install -r requirements.txt

import json
import time
import urllib.parse
import urllib.request
from pathlib import Path

import geopandas as gpd
import pandas as pd


PROJECT_DIR = REPO_DIR
RESULTS_DIR = PROJECT_DIR / 'outputs/grand_canyon_v2'
GEOLOGY_DIR = PROJECT_DIR / 'outputs/geology'
GEOLOGY_DIR.mkdir(parents=True, exist_ok=True)
GEOLOGY_FILE = GEOLOGY_DIR / 'grand_canyon_geology.geojson'
DETECTIONS_IN = RESULTS_DIR / 'grand_canyon_detections.geojson'
DETECTIONS_OUT = RESULTS_DIR / 'grand_canyon_detections_geology.geojson'

SERVICE = (
    'https://services.arcgis.com/v01gqwM5QqNysAAi/ArcGIS/rest/services/'
    'SB_5888bf4fe4b05ccb964bab9d_USGS_SGMC_feature/FeatureServer/3/query'
)
BBOX = '-114.25,35.45,-111.25,37.05'
FIELDS = 'UNIT_NAME,SGMC_LABEL,GENERALIZED_LITH,AGE_MIN,AGE_MAX,MAJOR1'
PAGE_SIZE = 500

features = []
offset = 0

while True:
    params = urllib.parse.urlencode({
        'where': '1=1',
        'geometry': BBOX,
        'geometryType': 'esriGeometryEnvelope',
        'inSR': 4326,
        'spatialRel': 'esriSpatialRelIntersects',
        'outFields': FIELDS,
        'outSR': 4326,
        'resultOffset': offset,
        'resultRecordCount': PAGE_SIZE,
        'f': 'geojson',
    })
    with urllib.request.urlopen(f'{SERVICE}?{params}', timeout=120) as response:
        page = json.load(response)
    rows = page.get('features', [])
    features.extend(rows)
    print('Downloaded:', len(features))
    if len(rows) < PAGE_SIZE:
        break
    offset += PAGE_SIZE

if not features:
    raise ValueError('No geology features were returned.')

geology = gpd.GeoDataFrame.from_features(features, crs=4326)
geology['geometry'] = geology.geometry.buffer(0)
geology = geology[geology.geometry.notna() & ~geology.geometry.is_empty].reset_index(drop=True)
geology.to_file(GEOLOGY_FILE, driver='GeoJSON')

detections = gpd.read_file(DETECTIONS_IN)
geology_utm = geology.to_crs('EPSG:32612')
detections_utm = detections.to_crs('EPSG:32612')
tree = geology_utm.sindex

units = []
lithologies = []
ages = []
shares = []

for geometry in detections_utm.geometry:
    best_index = None
    best_area = 0
    for index in tree.query(geometry, predicate='intersects'):
        area = geometry.intersection(geology_utm.geometry.iloc[index]).area
        if area > best_area:
            best_area = area
            best_index = index
    if best_index is None:
        units.append('Not available')
        lithologies.append('Not available')
        ages.append('Not available')
        shares.append(0)
    else:
        row = geology_utm.iloc[best_index]
        units.append(str(row.UNIT_NAME))
        lithologies.append(str(row.GENERALIZED_LITH))
        ages.append(f'{row.AGE_MAX} to {row.AGE_MIN}')
        shares.append(round(best_area / geometry.area, 3))

detections['geology_unit'] = units
detections['geology_lithology'] = lithologies
detections['geology_age'] = ages
detections['geology_share'] = shares
detections['scene_date'] = detections['scene_date'].astype(str)
detections.to_file(DETECTIONS_OUT, driver='GeoJSON')

# (The original cell then patched and restarted the Colab app; the app in this repository already reads this file.)
print('Geology units:', len(geology))
print('Lithology groups:', geology['GENERALIZED_LITH'].nunique())
print('Detections linked:', int((detections['geology_unit'] != 'Not available').sum()), 'of', len(detections))
print(detections['geology_lithology'].value_counts().head(8).to_string())
